# 02 – Preprocessing Pipeline
## ADSC21 Prüfungsleistung – Milwaukee Property Sales

**Ziel:** Aufbau einer vollständigen scikit-learn Pipeline für numerische und kategorielle Features als Grundlage für das Modelltraining.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_theme(style='whitegrid')
print('Libraries loaded.')

## 1. Daten laden (bereinigter Datensatz aus Notebook 01)

In [ ]:
df = pd.read_csv('../data/property_sales_clean.csv')
print(f'Shape: {df.shape}')
print(f'Spalten: {df.columns.tolist()}')
df.head()

In [ ]:
# Zielvariable bestimmen
price_col = 'sale_price' if 'sale_price' in df.columns else 'saleprice'

# Feature-Typen definieren
numerical_features = [c for c in ['fin_sqft', 'year_built', 'bedrooms', 'bathrooms', 'lotsize', 'stories']
                      if c in df.columns]
categorical_features = [c for c in ['style', 'extwall', 'zip_code'] if c in df.columns]

print('Numerische Features:', numerical_features)
print('Kategorielle Features:', categorical_features)
print('Zielvariable:', price_col)

## 2. Train / Test Split

In [ ]:
X = df[numerical_features + categorical_features]
y = df[price_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training:  {X_train.shape[0]:,} Zeilen')
print(f'Test:      {X_test.shape[0]:,} Zeilen')

## 3. Preprocessing Pipelines

Wir bauen separate Pipelines für numerische und kategorielle Merkmale
und kombinieren sie mit einem `ColumnTransformer` (vgl. Kurs Sektion 2).

In [ ]:
# --- Numerische Pipeline ---
# RobustScaler ist weniger empfindlich gegenüber Ausreißern als StandardScaler
numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])

# --- Kategorielle Pipeline ---
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(
        handle_unknown='ignore',
        sparse_output=False,
        min_frequency=10  # seltene Kategorien als 'infrequent' zusammenfassen
    ))
])

# --- ColumnTransformer ---
preprocessor = ColumnTransformer([
    ('num', numerical_pipeline, numerical_features),
    ('cat', categorical_pipeline, categorical_features)
], remainder='drop')

print('Preprocessor definiert.')
print(preprocessor)

## 4. Baseline-Modell: Lineare Regression

In [ ]:
baseline_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

baseline_pipeline.fit(X_train, y_train)
y_pred_baseline = baseline_pipeline.predict(X_test)

rmse_baseline = np.sqrt(mean_squared_error(y_test, y_pred_baseline))
mae_baseline  = mean_absolute_error(y_test, y_pred_baseline)
r2_baseline   = r2_score(y_test, y_pred_baseline)

print('=== Baseline: Lineare Regression ===')
print(f'RMSE: ${rmse_baseline:,.0f}')
print(f'MAE:  ${mae_baseline:,.0f}')
print(f'R²:   {r2_baseline:.4f}')

## 5. PCA-Erweiterung der Pipeline (Exkurs)

In [ ]:
# Nur auf numerische Features angewendet, um interpretierbar zu bleiben
numerical_pipeline_pca = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler()),
    ('pca', PCA(n_components=0.95, random_state=42))  # 95% erklärte Varianz
])

preprocessor_pca = ColumnTransformer([
    ('num', numerical_pipeline_pca, numerical_features),
    ('cat', categorical_pipeline, categorical_features)
], remainder='drop')

pca_pipeline = Pipeline([
    ('preprocessor', preprocessor_pca),
    ('model', LinearRegression())
])

pca_pipeline.fit(X_train, y_train)
y_pred_pca = pca_pipeline.predict(X_test)

rmse_pca = np.sqrt(mean_squared_error(y_test, y_pred_pca))
r2_pca   = r2_score(y_test, y_pred_pca)

n_components = pca_pipeline.named_steps['preprocessor'].transformers_[0][1].named_steps['pca'].n_components_

print(f'=== Lineare Regression + PCA (95% Varianz, {n_components} Komponenten) ===')
print(f'RMSE: ${rmse_pca:,.0f}')
print(f'R²:   {r2_pca:.4f}')
print(f'\nBaseline RMSE: ${rmse_baseline:,.0f} | PCA RMSE: ${rmse_pca:,.0f}')

## 6. Feature-Matrix visualisieren

In [ ]:
# Transformierten Datensatz anschauen
X_train_transformed = preprocessor.fit_transform(X_train)
print(f'Eingabe-Shape:  {X_train.shape}')
print(f'Ausgabe-Shape:  {X_train_transformed.shape}')
print(f'Neue Dimensionen durch One-Hot-Encoding: {X_train_transformed.shape[1] - len(numerical_features)}')

In [ ]:
# Predicted vs Actual – Baseline
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_pred, title in zip(axes,
                              [y_pred_baseline, y_pred_pca],
                              ['Lineare Regression (Baseline)', 'Lineare Regression + PCA']):
    ax.scatter(y_test / 1000, y_pred / 1000, alpha=0.2, s=8, color='steelblue')
    lims = [min(y_test.min(), np.min(y_pred)) / 1000,
            max(y_test.max(), np.max(y_pred)) / 1000]
    ax.plot(lims, lims, 'r--', lw=1.5)
    ax.set_xlabel('Tatsächlicher Preis (Tsd. USD)')
    ax.set_ylabel('Vorhergesagter Preis (Tsd. USD)')
    ax.set_title(title, fontsize=11)

plt.tight_layout()
plt.savefig('../report/abb6_predicted_vs_actual_baseline.png', bbox_inches='tight')
plt.show()

## 7. Pipeline speichern

In [ ]:
joblib.dump(baseline_pipeline, '../data/baseline_pipeline.pkl')
print('Baseline Pipeline gespeichert: data/baseline_pipeline.pkl')

# Metadaten speichern
import json
meta = {
    'numerical_features': numerical_features,
    'categorical_features': categorical_features,
    'target': price_col,
    'baseline_rmse': float(rmse_baseline),
    'baseline_mae': float(mae_baseline),
    'baseline_r2': float(r2_baseline)
}
with open('../data/meta.json', 'w') as f:
    json.dump(meta, f, indent=2)
print('Metadaten gespeichert:', meta)